![interpreto_banner](../assets/img/interpreto_banner.png){ style="display:block; max-width:100%; height:auto; margin:0 auto;" }

# Generation Demonstration

This notebook show case what can be done with `interpreto` for generation.

*author: Antonin Poché*

## 0. Imports and model loading

In [1]:
!uv pip install interpreto datasets

Using Python 3.12.3 environment at: /home/antonin.poche/interpreto/.venv
Audited 2 packages in 190ms


In [2]:
import os

import datasets
import transformers

import interpreto
from interpreto import KernelShap, plot_attributions, plot_concepts
from interpreto.concepts import LLMLabels, SemiNMFConcepts
from interpreto.model_wrapping.llm_interface import OpenAILLM

In [3]:
# load dataset examples
dataset = datasets.load_dataset("dair-ai/emotion", "split")["train"]["text"][:1000]

In [4]:
# load the model and tokenizer
tokenizer = transformers.AutoTokenizer.from_pretrained("gpt2")
model = transformers.AutoModelForCausalLM.from_pretrained("gpt2").cuda()

## 1. Attribution Demonstration

In [5]:
# instantiate Lime
attribution_explainer = KernelShap(model, tokenizer)

# compute attributions
attributions = attribution_explainer(
    model_inputs="Alice and Bob enter the bar, ",
    targets="then Alice offers a drink to Bob.",
)

# visualize attributions
plot_attributions(attributions[0])

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


## 2. Concepts Demonstration

### 2.1 Split model and get a dataset of activations

In [6]:
# split the model
split_model = interpreto.ModelWithSplitPoints(model, tokenizer=tokenizer, split_points=[8])

# compute the token activations
granularity = split_model.activation_granularities.TOKEN
activations = split_model.get_activations(
    inputs=dataset,
    activation_granularity=granularity,
)

### 2.2 Learn concepts as patterns in the activations

In [7]:
# create an explainer around the split model
concept_explainer = SemiNMFConcepts(split_model, nb_concepts=20)

# train the concept model of the explainer
concept_explainer.fit(activations)

### 2.3 Interpret the concepts

In [8]:
llm_interface = OpenAILLM(api_key=os.getenv("OPENAI_API_KEY"), model="gpt-4.1-nano")

# instantiate the interpretation method with the concept explainer
interpretation_method = LLMLabels(
    concept_explainer=concept_explainer,
    activation_granularity=granularity,
    llm_interface=llm_interface,
    k_examples=20,
)

# interpret the concepts via top-k words
llm_labels = interpretation_method.interpret(
    inputs=dataset,
    latent_activations=activations,
    concepts_indices="all",
)

In [9]:
print("\n".join(llm_labels.values()))

Repetition of "feel" at high importance
Repetition of lowercase "i" variations
Fragmented, generic, pronoun-heavy language
Repetition of first-person pronouns with slight variations.
Pronoun repetition with slight variations
Lowercase "i" with varying but predominantly high confidence scores.
Contextual, descriptive adjectives and pronouns
Simple pronoun usage
Repetitive single-letter patterns
Contraction variations of "I"
Single-letter dominance
Self-referential minimalism
Single-letter dominance
Single-letter and common words with repeated emphasis
Single-letter dominance
Self-referential pronoun concentration
High-frequency single-letter "i" repetitions
Single-letter dominance
Variations of first-person pronouns
Single-letter repetitions with informal contractions
